# 01 - LangGraph：构建复杂 Agent 工作流

## 学习目标

- 理解 LangGraph 的核心概念：StateGraph、Node、Edge
- 掌握状态管理和条件路由
- 学习构建循环、分支、并行工作流
- 实现一个完整的 LangGraph Agent

---

## 1. LangGraph 概述

### 1.1 为什么需要 LangGraph？

LangChain 的 Chain 是**线性**的，难以表达复杂的控制流：

```
Chain: A → B → C → D (线性，无法循环)

实际场景需要：
- 循环：A → B → (条件判断) → A (迭代优化)
- 分支：A → (判断) → B 或 C (条件路由)
- 并行：A → [B, C] → D (并行执行)
- 状态：在步骤间传递和更新状态
```

**LangGraph** 通过**状态图（StateGraph）**解决这些问题。

### 1.2 核心概念

| 概念 | 描述 | 类比 |
|------|------|------|
| **State** | 共享状态对象，在各节点间传递 | 全局变量 / Redux Store |
| **Node** | 处理函数，接收状态并返回更新 | 函数 / 组件 |
| **Edge** | 连接节点，定义流转规则 | 跳转 / 路由 |
| **Graph** | 完整的流程定义 | 程序 / 工作流 |

### 1.3 LangGraph 架构图

```
┌─────────────────────────────────────────────────────────────┐
│                     LangGraph 架构                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────┐      ┌─────────┐      ┌─────────┐            │
│  │  Node A │─────►│  Node B │─────►│  Node C │            │
│  │ (入口)  │      │ (处理)  │      │ (判断)  │            │
│  └─────────┘      └─────────┘      └────┬────┘            │
│                                         │                 │
│                    ┌────────────────────┘                 │
│                    │                                      │
│                    ▼ (条件: 未完成)                        │
│              ┌─────────┐                                  │
│              │  Node D │ (优化)                           │
│              └────┬────┘                                  │
│                   │                                       │
│                   └────────────────► (回到 Node B)         │
│                                                             │
│                    (条件: 完成)                             │
│                    ▼                                        │
│              ┌─────────┐                                   │
│              │  Node E │ (输出)                            │
│              └─────────┘                                   │
│                                                             │
│  状态 (State) 在节点间传递和更新                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---

## 2. 基础概念详解

### 2.1 State（状态）

状态是 LangGraph 的核心，所有节点共享同一个状态对象。



---
## 0. 模型准备：加载 Qwen2.5-7B-Instruct

本 Notebook 使用 **ModelScope** 加载本地 Qwen 模型，替代在线 API 调用。

- **推荐模型**：`Qwen/Qwen2.5-7B-Instruct`（约 15GB 显存）
- **低显存备选**：`Qwen/Qwen2.5-3B-Instruct`（约 6GB 显存）
- 自动检测 GPU / CPU，优先使用 GPU 加速

> 如果没有安装 modelscope 或显存不足，可以使用下方代码中的 **MockLLM 备选方案**。



In [ ]:
# ============================================================
# 安装依赖（首次运行时取消注释）
# ============================================================
# !pip install modelscope torch transformers -q

import torch

# ============================================================
# GPU / CPU 自动检测
# ============================================================
if torch.cuda.is_available():
    DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"检测到 GPU: {GPU_NAME}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("检测到 Apple Silicon GPU (MPS)")
else:
    DEVICE = "cpu"
    print("未检测到 GPU，将使用 CPU（速度较慢）")

# ============================================================
# QwenLLM 封装类
# ============================================================
from modelscope import AutoModelForCausalLM, AutoTokenizer

class QwenLLM:
    """基于 ModelScope 的 Qwen 模型封装"""

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        # 自动检测设备
        if device is None:
            device = "cuda" if torch.cuda.is_available() else (
                "mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
                else "cpu"
            )
        self.device = device
        print(f"正在加载模型 {model_name}，设备: {device} ...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype="auto", device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.messages = []
        print("模型加载完成！")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """对话接口"""
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})
        return response

    def reset(self):
        """清空对话历史"""
        self.messages = []

# ============================================================
# 初始化模型
# ============================================================
# 低显存环境可切换为: Qwen/Qwen2.5-3B-Instruct
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

try:
    llm = QwenLLM(model_name=MODEL_NAME)
    USE_REAL_MODEL = True
    print("\n使用真实 Qwen 模型")
except Exception as e:
    print(f"\n模型加载失败: {e}")
    print("将使用 MockLLM 备选方案")
    USE_REAL_MODEL = False

print(f"\n当前设备: {DEVICE}")
print(f"使用真实模型: {USE_REAL_MODEL}")



In [ ]:
# ============================================================
# 无模型时的备选方案：MockLLM
# （仅在上方模型加载失败时使用）
# ============================================================

if not USE_REAL_MODEL:
    class MockLLM:
        """模拟 LLM，用于无模型环境下的教学演示"""

        def __init__(self, model_name="mock-qwen"):
            self.model_name = model_name
            self.messages = []

        def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
            """模拟对话回复"""
            if '你好' in user_message:
                response = "你好！我是 AI 助手，有什么可以帮助你的吗？"
            elif '天气' in user_message:
                response = "我无法获取实时天气信息，建议您查看天气应用。"
            elif 'LangChain' in user_message:
                response = "LangChain 是一个用于开发 LLM 应用的框架，提供了 Chains、Prompts、Memory 等核心组件。"
            elif 'Agent' in user_message:
                response = "Agent 是一种能够自主决策并调用工具的智能体，通常使用 ReAct 模式工作。"
            else:
                response = f"我收到了您的消息：'{user_message[:50]}'。这是一个模拟回复。"

            self.messages.append({"role": "user", "content": user_message})
            self.messages.append({"role": "assistant", "content": response})
            return response

        def reset(self):
            self.messages = []

    llm = MockLLM(model_name="mock-qwen")
    print("已启用 MockLLM 备选方案")

# 测试模型调用
response = llm.chat("你好，请介绍一下自己")
print(f"模型回复: {response}")



In [ ]:
# 状态定义示例
from typing import TypedDict, List, Annotated
from operator import add

# 定义状态结构
class AgentState(TypedDict):
    """
    Agent 状态定义

    所有字段都是可选的，根据需要在节点间传递
    """
    # 输入
    query: str  # 用户查询

    # 中间状态
    messages: Annotated[List[dict], add]  # 消息历史（使用 add 合并）
    current_step: str  # 当前步骤

    # 工具调用
    tool_calls: List[dict]  # 待执行的工具调用
    tool_results: List[dict]  # 工具执行结果

    # 输出
    answer: str  # 最终答案

    # 控制流
    iteration_count: int  # 迭代计数
    is_complete: bool  # 是否完成

# 创建初始状态
initial_state = AgentState(
    query="今天北京的天气怎么样？",
    messages=[],
    current_step="start",
    tool_calls=[],
    tool_results=[],
    answer="",
    iteration_count=0,
    is_complete=False
)

print("状态定义完成")
print(f"初始状态: {initial_state}")



### 2.2 Node（节点）

节点是处理函数，接收当前状态，返回状态更新。



In [ ]:
# 节点函数示例（使用 QwenLLM 生成回答）

def understand_query(state: AgentState) -> AgentState:
    """
    节点 1：理解用户查询

    分析用户意图，确定需要什么工具
    """
    query = state['query']

    print(f"[Node: understand_query] 分析查询: {query}")

    # 使用 QwenLLM 进行意图识别
    intent_prompt = f"请分析以下用户查询的意图，判断需要使用哪个工具（weather_tool/calculator_tool/general_tool）。只回答工具名称。\n\n用户查询：{query}"
    try:
        tool_name = llm.chat(intent_prompt, max_new_tokens=50).strip()
        # 确保返回合法的工具名
        valid_tools = ['weather_tool', 'calculator_tool', 'general_tool']
        if tool_name not in valid_tools:
            tool_name = 'general_tool'
    except Exception:
        # 备选：简单关键词匹配
        if '天气' in query:
            tool_name = 'weather_tool'
        elif '计算' in query or '等于' in query:
            tool_name = 'calculator_tool'
        else:
            tool_name = 'general_tool'

    return {
        'messages': [{
            'role': 'system',
            'content': f'识别到意图，需要使用工具: {tool_name}'
        }],
        'current_step': 'understood',
        'tool_calls': [{'tool': tool_name, 'params': query}]
    }

def execute_tools(state: AgentState) -> AgentState:
    """
    节点 2：执行工具

    调用外部工具获取信息
    """
    tool_calls = state.get('tool_calls', [])
    results = []

    print(f"[Node: execute_tools] 执行工具调用")

    for call in tool_calls:
        tool_name = call['tool']

        # 模拟工具执行
        if tool_name == 'weather_tool':
            result = "北京：晴天，25°C，空气质量良"
        elif tool_name == 'calculator_tool':
            result = "计算结果: 42"
        else:
            result = "通用回答: 已处理请求"

        results.append({
            'tool': tool_name,
            'result': result
        })
        print(f"  - {tool_name}: {result}")

    return {
        'tool_results': results,
        'current_step': 'tools_executed'
    }

def generate_answer(state: AgentState) -> AgentState:
    """
    节点 3：生成回答（使用 QwenLLM）

    基于工具结果，调用 QwenLLM 生成最终答案
    """
    tool_results = state.get('tool_results', [])
    query = state['query']

    print(f"[Node: generate_answer] 生成回答")

    # 构建提示词
    context_str = "\n".join([f"- {r['tool']}: {r['result']}" for r in tool_results])
    prompt = f"用户问题：{query}\n\n工具返回结果：\n{context_str}\n\n请根据以上信息，用中文给出简洁准确的回答。"

    try:
        # 使用 QwenLLM 生成回答
        answer = llm.chat(prompt, max_new_tokens=256)
    except Exception:
        # 备选：简单拼接
        if tool_results:
            answer = f"根据查询结果：{tool_results[0]['result']}"
        else:
            answer = f"关于 '{query}'，我暂时无法获取相关信息。"

    return {
        'answer': answer,
        'messages': [{
            'role': 'assistant',
            'content': answer
        }],
        'current_step': 'answered',
        'is_complete': True
    }

def check_quality(state: AgentState) -> AgentState:
    """
    节点 4：质量检查

    检查回答质量，决定是否需要重新生成
    """
    answer = state.get('answer', '')
    iteration = state.get('iteration_count', 0)

    print(f"[Node: check_quality] 质量检查 (迭代 {iteration + 1})")

    # 模拟质量检查
    is_good = len(answer) > 10 and iteration < 2

    return {
        'iteration_count': iteration + 1,
        'is_complete': is_good
    }

print("节点函数定义完成")
print("定义的节点:")
print("  - understand_query: 理解查询（使用 QwenLLM）")
print("  - execute_tools: 执行工具")
print("  - generate_answer: 生成回答（使用 QwenLLM）")
print("  - check_quality: 质量检查")



### 2.3 Edge（边）和条件路由

边定义节点之间的连接，支持条件路由。



In [ ]:
# 条件路由函数

def route_after_check(state: AgentState) -> str:
    """
    条件路由：根据质量检查结果决定下一步

    Returns:
        'complete' - 质量通过，结束
        'regenerate' - 需要重新生成
    """
    is_complete = state.get('is_complete', False)

    if is_complete:
        print(f"[Router] 质量检查通过 → 结束")
        return 'complete'
    else:
        print(f"[Router] 质量检查未通过 → 重新生成")
        return 'regenerate'

print("条件路由函数定义完成")



---

## 3. 构建完整的 StateGraph

将节点和边组合成完整的图。



In [ ]:
# 模拟 StateGraph 实现

class MockStateGraph:
    """模拟 LangGraph 的 StateGraph"""

    def __init__(self, state_type):
        self.state_type = state_type
        self.nodes = {}
        self.edges = {}
        self.conditional_edges = {}
        self.entry_point = None

    def add_node(self, name, func):
        """添加节点"""
        self.nodes[name] = func
        print(f"[Graph] 添加节点: {name}")

    def add_edge(self, from_node, to_node):
        """添加普通边"""
        if from_node not in self.edges:
            self.edges[from_node] = []
        self.edges[from_node].append(to_node)
        print(f"[Graph] 添加边: {from_node} → {to_node}")

    def add_conditional_edges(
        self,
        from_node,
        condition_func,
        path_map
    ):
        """添加条件边"""
        self.conditional_edges[from_node] = {
            'condition': condition_func,
            'paths': path_map
        }
        print(f"[Graph] 添加条件边: {from_node} → {path_map}")

    def set_entry_point(self, node_name):
        """设置入口节点"""
        self.entry_point = node_name
        print(f"[Graph] 设置入口: {node_name}")

    def compile(self):
        """编译图"""
        print(f"\n[Graph] 编译完成")
        print(f"[Graph] 节点: {list(self.nodes.keys())}")
        return CompiledGraph(self)

class CompiledGraph:
    """编译后的图"""

    def __init__(self, graph):
        self.graph = graph

    def invoke(self, initial_state):
        """执行图"""
        state = dict(initial_state)
        current_node = self.graph.entry_point
        visited = []

        print(f"\n{'='*60}")
        print("开始执行 StateGraph")
        print(f"{'='*60}")

        max_steps = 20
        step = 0

        while current_node and step < max_steps:
            step += 1
            visited.append(current_node)

            print(f"\nStep {step}: 执行节点 '{current_node}'")
            print("-" * 40)

            # 执行节点
            if current_node in self.graph.nodes:
                node_func = self.graph.nodes[current_node]
                updates = node_func(state)

                # 更新状态
                for key, value in updates.items():
                    if key in state and isinstance(value, list) and isinstance(state[key], list):
                        # 合并列表（使用 Annotated 语义）
                        state[key] = state[key] + value
                    else:
                        state[key] = value

            # 确定下一个节点
            if current_node in self.graph.conditional_edges:
                # 条件路由
                edge_info = self.graph.conditional_edges[current_node]
                result = edge_info['condition'](state)
                next_node = edge_info['paths'].get(result)
                print(f"\n[条件路由] {current_node} → {next_node} (条件: {result})")
                current_node = next_node
            elif current_node in self.graph.edges:
                # 普通边
                next_nodes = self.graph.edges[current_node]
                current_node = next_nodes[0] if next_nodes else None
            else:
                current_node = None

            # 检查是否完成
            if state.get('is_complete') and current_node is None:
                print(f"\n任务完成")
                break

        print(f"\n{'='*60}")
        print(f"执行统计")
        print(f"{'='*60}")
        print(f"执行节点: {' → '.join(visited)}")
        print(f"总步数: {step}")

        return state

# 创建图
workflow = MockStateGraph(AgentState)

# 添加节点
workflow.add_node("understand", understand_query)
workflow.add_node("execute_tools", execute_tools)
workflow.add_node("generate", generate_answer)
workflow.add_node("check", check_quality)

# 添加边
workflow.add_edge("understand", "execute_tools")
workflow.add_edge("execute_tools", "generate")
workflow.add_edge("generate", "check")

# 添加条件边
workflow.add_conditional_edges(
    "check",
    route_after_check,
    {
        "complete": None,  # 结束
        "regenerate": "generate"  # 重新生成
    }
)

# 设置入口
workflow.set_entry_point("understand")

# 编译
app = workflow.compile()



In [ ]:
# 执行图
result = app.invoke(initial_state)

print(f"\n最终答案: {result.get('answer')}")
print(f"消息历史: {len(result.get('messages', []))} 条")



---

## 4. 实际使用 LangGraph

### 4.1 安装和基础使用



In [ ]:
# 实际 LangGraph 代码示例（注释形式展示）

langgraph_example = '''
# 安装
# pip install langgraph

from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
import operator

# 定义状态
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    next_step: str

# 创建图
workflow = StateGraph(AgentState)

# 定义节点
def agent_node(state):
    """Agent 决策节点"""
    messages = state['messages']
    # 调用 LLM 决定下一步
    response = llm.invoke(messages)
    return {'messages': [response]}

# 添加节点
workflow.add_node('agent', agent_node)
workflow.add_node('tools', ToolNode(tools))

# 添加边
workflow.add_edge('agent', 'tools')
workflow.add_edge('tools', 'agent')

# 设置入口
workflow.set_entry_point('agent')

# 编译
app = workflow.compile()

# 执行
result = app.invoke({
    'messages': [HumanMessage(content='今天天气怎么样？')]
})
'''

print("实际 LangGraph 代码示例：")
print("=" * 50)
print(langgraph_example)



### 4.2 常见模式

#### 模式 1：ReAct Agent



In [ ]:
react_pattern = '''
from langgraph.prebuilt import create_react_agent

# 一键创建 ReAct Agent
agent = create_react_agent(
    model=llm,
    tools=tools,
    messages_modifier="你是一个 helpful 的助手"
)

# 执行
result = agent.invoke(
    {"messages": [("human", "今天天气怎么样？")]}
)
'''

print("ReAct Agent 模式：")
print(react_pattern)



#### 模式 2：带循环的工作流



In [ ]:
loop_pattern = '''
# 带循环的工作流：迭代优化

def should_continue(state):
    """判断是否继续迭代"""
    messages = state['messages']
    last_message = messages[-1]

    # 如果 AI 说任务完成，则结束
    if '任务完成' in last_message.content:
        return 'end'

    # 否则继续
    return 'continue'

# 创建图
workflow = StateGraph(AgentState)

workflow.add_node('agent', agent_node)
workflow.add_node('action', action_node)

# 条件边
workflow.add_conditional_edges(
    'agent',
    should_continue,
    {
        'continue': 'action',
        'end': END
    }
)

workflow.add_edge('action', 'agent')
workflow.set_entry_point('agent')

app = workflow.compile()
'''

print("带循环的工作流模式：")
print(loop_pattern)



#### 模式 3：并行执行



In [ ]:
parallel_pattern = '''
# 并行执行多个任务

from langgraph.constants import Send

def distribute_tasks(state):
    """分发任务到多个并行节点"""
    topics = state['topics']

    # 为每个 topic 创建一个并行任务
    return [
        Send('process_topic', {'topic': topic})
        for topic in topics
    ]

def process_topic(state):
    """处理单个 topic"""
    topic = state['topic']
    # 处理逻辑...
    return {'results': [f'{topic} 处理完成']}

def aggregate(state):
    """聚合结果"""
    results = state['results']
    return {'final': '\n'.join(results)}

# 创建图
workflow = StateGraph(AgentState)

workflow.add_node('distribute', distribute_tasks)
workflow.add_node('process_topic', process_topic)
workflow.add_node('aggregate', aggregate)

# 条件边：一个节点分发到多个并行节点
workflow.add_conditional_edges('distribute', distribute_tasks)

# 所有并行节点完成后汇聚
workflow.add_edge('process_topic', 'aggregate')

workflow.set_entry_point('distribute')
app = workflow.compile()
'''

print("并行执行模式：")
print(parallel_pattern)



---

## 5. LangGraph vs LangChain

| 特性 | LangChain | LangGraph |
|------|-----------|-----------|
| **控制流** | 线性 Chain | 任意图结构 |
| **循环** | 不支持 | 原生支持 |
| **条件分支** | 有限 | 完整支持 |
| **并行** | 不支持 | 支持 |
| **状态管理** | 简单 Memory | 完整 State |
| **适用场景** | 简单流程 | 复杂 Agent |
| **学习曲线** | 较低 | 较高 |

---

## 6. 小结

### 核心要点

1. **LangGraph** 基于 StateGraph 构建复杂 Agent 工作流
2. **核心概念**：State（状态）、Node（节点）、Edge（边）
3. **支持循环**：Agent 可以迭代优化直到满足条件
4. **条件路由**：根据状态动态决定执行路径
5. **并行执行**：同时处理多个独立任务

### 下一步

- [02_autogen_multi_agent.ipynb](02_autogen_multi_agent.ipynb) - 学习 AutoGen 多 Agent 框架
- [03_crewai_collaboration.ipynb](03_crewai_collaboration.ipynb) - 探索 CrewAI 协作框架

---

## 参考资源

- [LangGraph 官方文档](https://langchain-ai.github.io/langgraph/)
- [LangGraph 概念指南](https://langchain-ai.github.io/langgraph/concepts/)
- [LangGraph 教程](https://langchain-ai.github.io/langgraph/tutorials/)

